In [16]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string as idx

# 1. Configurações
caminho = r"C:\Users\FERNANDO\Documents\Projetos\Mercadinho_ERP\1base_mestra\Base_de_dados_mercadinho - Copia.xlsx"
aba = "MOVIMENTACAO"
cab = 1  # Linha do cabeçalho no Excel (1-based)

# 2. Carrega com Pandas apenas para leitura rápida
df = pd.read_excel(caminho, sheet_name=aba, header=cab - 1)

# Mapeia os nomes reais das colunas
col_D = df.columns[idx("D") - 1]
col_E = df.columns[idx("E") - 1]
col_A = df.columns[idx("A") - 1]
col_G = df.columns[idx("G") - 1]
col_M = df.columns[idx("M") - 1]
col_N = df.columns[idx("N") - 1]

# 3. Processamento Linha a Linha (Lógica Sequencial)
id_contador = 1
alvos_calculados = []

prev_d = None
prev_e = None
ref_g = None
ref_m = None
ref_n = None
ref_id = None

for i, row in df.iterrows():
    curr_d = row[col_D]
    curr_e = row[col_E]
    
    # Se for a primeira linha OU se D/E mudaram em relação à linha anterior
    if i == 0 or (curr_d != prev_d or curr_e != prev_e):
        ref_id = f"M{id_contador:05d}"
        id_contador += 1
        
        # Pega os valores originais desta linha como nova referência
        ref_g = row[col_G]
        ref_m = row[col_M]
        ref_n = row[col_N]
        
        # Atualiza a memória da linha anterior
        prev_d = curr_d
        prev_e = curr_e
    
    # Se D e E forem exatamente iguais à linha anterior,
    # ele mantém o ref_id, ref_g, ref_m, ref_n da referência acima.
    
    alvos_calculados.append({
        'A': ref_id,
        'G': ref_g,
        'M': ref_m,
        'N': ref_n
    })

df_resultado = pd.DataFrame(alvos_calculados)

# 4. Escreve EXCLUSIVAMENTE nas colunas A, G, M e N do Excel
wb = load_workbook(caminho)
ws = wb[aba]

linha_inicial_excel = cab + 1
col_letras = ["A", "G", "M", "N"]

for i, row in df_resultado.iterrows():
    linha_excel = linha_inicial_excel + i
    
    for letra in col_letras:
        valor = row[letra]
        val_final = None if pd.isna(valor) else valor
        # Alterando SOMENTE as colunas A, G, M e N no Excel
        ws.cell(row=linha_excel, column=idx(letra), value=val_final)

wb.save(caminho)
print("Sucesso! Colunas A, G, M e N atualizadas sequencialmente. D e E intactas!")

Sucesso! Colunas A, G, M e N atualizadas sequencialmente. D e E intactas!
